In [1]:
import pandas as pd
import numpy as np


import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv("twitter_training.csv")

In [3]:
df

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...
...,...,...,...,...
74676,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74677,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74678,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74679,9200,Nvidia,Positive,Just realized between the windows partition of...


In [4]:
df.columns = ["ID", "Entity", "Sentiment", "Tweet"]

In [5]:
df.head()

,ID,Entity,Sentiment,Tweet
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [6]:
df.shape

(74681, 4)

In [7]:
df.isnull().sum()

ID             0
Entity         0
Sentiment      0
Tweet        686
dtype: int64

In [8]:
df = df.dropna()

In [9]:
df.isnull().sum()

ID           0
Entity       0
Sentiment    0
Tweet        0
dtype: int64

In [10]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\idris\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
ps = PorterStemmer()

In [ ]:
df = df[df["Sentiment"] != "Irrelevant"]

print("\nSentiment Counts")
print(df["Sentiment"].value_counts())



ps = PorterStemmer()

def clean_text(text):

    
    text = re.sub('[^a-zA-Z]', ' ', text)

    
    text = text.lower()

  
    words = text.split()

    
    words = [ps.stem(word)
             for word in words
             if word not in stopwords.words('english')]

    
    text = " ".join(words)

    return text


df["Clean_Tweet"] = df["Tweet"].apply(clean_text)

print("\nCleaned Tweets")
print(df[["Tweet", "Clean_Tweet"]].head())



tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df["Clean_Tweet"])

y = df["Sentiment"]

print("\nShape of Feature Matrix")
print(X.shape)



X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining Data Shape")
print(X_train.shape)

print("\nTesting Data Shape")
print(X_test.shape)

svm = SVC(kernel='linear')

svm.fit(X_train, y_train)

print("\nModel Training Completed")


y_pred = svm.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy")
print(accuracy)

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))


while True:

    review = input("\nEnter Tweet (Type exit to stop): ")

    if review.lower() == "exit":
        break

    review = clean_text(review)

    review = tfidf.transform([review])

    prediction = svm.predict(review)

    print("Predicted Sentiment:", prediction[0])


Sentiment Counts
Sentiment
Negative    22358
Positive    20654
Neutral     18108
Name: count, dtype: int64

Cleaned Tweets
                                               Tweet  \
0  I am coming to the borders and I will kill you...   
1  im getting on borderlands and i will kill you ...   
2  im coming on borderlands and i will murder you...   
3  im getting on borderlands 2 and i will murder ...   
4  im getting into borderlands and i can murder y...   

                 Clean_Tweet  
0           come border kill  
1     im get borderland kill  
2  im come borderland murder  
3   im get borderland murder  
4   im get borderland murder  

Shape of Feature Matrix
(61120, 5000)

Training Data Shape
(48896, 5000)

Testing Data Shape
(12224, 5000)

Model Training Completed

Accuracy
0.7819862565445026

Classification Report
              precision    recall  f1-score   support

    Negative       0.79      0.83      0.81      4444
     Neutral       0.80      0.69      0.74      3679
    